In [1]:
import torch
from torch import nn


def conv_bn(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True)
    )


class Residual(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(conv_bn(c, c), conv_bn(c, c))

    def forward(self, x):
        return x + self.block(x)


class GalAxy10(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            conv_bn(3, 64),
            conv_bn(64, 128),
            nn.MaxPool2d(2),
            Residual(128),
            conv_bn(128, 256),
            nn.MaxPool2d(2),
            Residual(256),
            conv_bn(256, 512),
            nn.MaxPool2d(2),
            Residual(512),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(512, 10)
        )

    def forward(self, x: torch.Tensor):
        x = self.features(x)
        return self.classifier(x)

    

In [ ]:
!git clone https://github.com/Salman-Elhaddadi/Galaxy10-research /kaggle/working/repo
%cd /kaggle/working/repo
!mkdir -p data/raw
!wget -q -O data/raw/Galaxy10_DECals.h5 https://zenodo.org/records/10845026/files/Galaxy10_DECals.h5
!ls -lh data/raw
!python scripts/make_dataset.py
!python scripts/compute_normalisation.py
import sys; sys.path.insert(0, "/kaggle/working/repo/src")

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 89 (delta 18), reused 83 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 16.25 MiB | 20.34 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/kaggle/working/repo


In [7]:
%%writefile /kaggle/working/repo/src/galaxy10/model/model_a.py
import torch
from torch import nn


def conv_bn_relu(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
    )


class ModelA(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            conv_bn_relu(3, 32),
            nn.MaxPool2d(2),
            conv_bn_relu(32, 64),
            nn.MaxPool2d(2),
            conv_bn_relu(64, 128),
            nn.MaxPool2d(2),
            conv_bn_relu(128, 256),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

Writing /kaggle/working/repo/src/galaxy10/model/model_a.py


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/repo/src/galaxy10/model/model_a.py'

In [5]:
from pathlib import Path
import sys
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

sys.path.insert(0, "/kaggle/working/repo/src")
from galaxy10.data.dataset import Galaxy10Dataset
from galaxy10.model.model_a import ModelA
sys.path.insert(0, "/kaggle/working/repo/src")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
SPLITS_DIR = Path("/kaggle/working/repo/splits")
BATCH_SIZE = 32
EPOCHS = 2
MAX_LR = 1e-3
EVAL_TEST = False

def train_tf(image):
    if torch.rand(()) < 0.5:
        image = torch.flip(image, dims=(2,))
    if torch.rand(()) < 0.5:
        image = torch.flip(image, dims=(1,))
    return torch.rot90(
        image, k=int(torch.randint(0, 4, ()).item()), dims=(1, 2)
    )


train_idx = np.load(SPLITS_DIR / "train_idx.npy")
val_idx = np.load(SPLITS_DIR / "val_idx.npy")
test_idx = np.load(SPLITS_DIR / "test_idx.npy")

train_ds = Galaxy10Dataset(train_idx, transform=train_tf)
val_ds = Galaxy10Dataset(val_idx)
test_ds = Galaxy10Dataset(test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

model = ModelA().to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=MAX_LR)

# Once, after creating the model and optimizer
model = model.to(device)
if device == "cuda":
    model = model.to(memory_format=torch.channels_last)

scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

def train_step(model:nn.Module,
               data_loader:DataLoader,
               loss_fn:nn.Module,
               optimizer:torch.optim.Optimizer,
               device: torch.device = device):

    train_loss, correct = 0, 0
    model.train()
    for X, y in data_loader:
        X, y = X.to(device), y.to(device)

        y_pred = model(X)
        loss = loss_fn(y_pred, y)

        train_loss += loss.item()
        correct += (y_pred.argmax(1) == y).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(data_loader)
    train_acc = correct / len(data_loader.dataset) * 100

    return train_loss, train_acc

def test_step(model:nn.Module,
              data_loader:DataLoader,
              loss_fn:nn.Module,
              device:torch.device = device):

    test_loss, correct = 0, 0
    preds, targets = [], []
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device), y.to(device)

            test_pred = model(X)
            test_loss += loss_fn(test_pred, y).item()
            correct += (test_pred.argmax(1) == y).sum().item()
            preds.append(test_pred.argmax(1).cpu())
            targets.append(y.cpu())

    test_loss /= len(data_loader)
    test_acc = correct / len(data_loader.dataset) * 100
    test_f1 = f1_score(torch.cat(targets).numpy(), torch.cat(preds).numpy(), average="macro") * 100

    return test_loss, test_acc, test_f1

best_val_f1 = 0.0
history = []
for epoch in range(EPOCHS):

     print(f"\nEpoch: {epoch + 1} / {EPOCHS}")

     train_loss, train_acc = train_step(
         model=model,
         data_loader=train_loader,
         loss_fn=loss_fn,
         optimizer=optimizer,
         device=device
     )

     val_loss, val_acc, val_f1 = test_step(
         model=model,
         data_loader=val_loader,
         loss_fn=loss_fn,
         device=device
     )

     history.append({
         "epoch": epoch + 1,
         "train_loss": train_loss,
         "train_acc": train_acc,
         "val_loss": val_loss,
         "val_acc": val_acc,
         "val_f1": val_f1,
     })

     if val_f1 > best_val_f1:
         best_val_f1 = val_f1
         torch.save(model.state_dict(), f"best_seed{SEED}.pt")

     print(
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_acc:.2f}% | "
        f"Val loss: {val_loss:.4f} | "
        f"Val accuracy: {val_acc:.2f}% | "
        f"Val macro-F1: {val_f1:.2f}%"
     )

with open(f"history_seed{SEED}.json", "w") as f:
    json.dump(history, f)

print(f"\nBest validation macro-F1: {best_val_f1:.2f}%")

if EVAL_TEST:
    model.load_state_dict(torch.load(f"best_seed{SEED}.pt", map_location=device))
    test_loss, test_acc, test_f1 = test_step(model, test_loader, loss_fn, device)
    print(f"Test accuracy: {test_acc:.2f}%")
    print(f"Test macro-F1: {test_f1:.2f}%")

cuda


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/repo/splits/train_idx.npy'

In [8]:
import glob, torch
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 0
path = glob.glob(f"/kaggle/working/**/best_seed{SEED}.pt", recursive=True)[0]
ckpt = torch.load(path, map_location=device)
state = ckpt.get("model_state_dict", ckpt.get("model", ckpt)) if isinstance(ckpt, dict) else ckpt
model.load_state_dict(state)
model.eval()

ys, ps = [], []
with torch.no_grad():
    for x, y in val_loader:
        ps.append(model(x.to(device)).argmax(1).cpu())
        ys.append(y)
y_true, y_pred = torch.cat(ys).numpy(), torch.cat(ps).numpy()

print(f"val macro-F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(classification_report(y_true, y_pred, digits=3))
cm = confusion_matrix(y_true, y_pred)
print((cm / cm.sum(1, keepdims=True)).round(2))

IndexError: list index out of range

In [23]:
m = ModelA().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
X, y = next(iter(train_loader))
X, y = X.to(device), y.to(device)

m.eval()
with torch.inference_mode():
    print("initial loss:", loss_fn(m(X), y).item())

m.train()
for step in range(201):
    opt.zero_grad()
    out = m(X)
    loss = loss_fn(out, y)
    loss.backward()
    opt.step()
    if step % 40 == 0:
        print(step, round(loss.item(), 4), (out.argmax(1) == y).float().mean().item())

initial loss: 2.3189778327941895
0 2.4029 0.03125
40 0.2402 1.0
80 0.02 1.0
120 0.008 1.0
160 0.0035 1.0
200 0.0031 1.0


In [19]:
import time
t = time.time()
for i, (X, y) in enumerate(train_loader):
    if i == 50:
        break
print("data only, 50 batches:", time.time() - t)

m = GalAxy10().to(device)
opt = torch.optim.Adam(m.parameters())
X, y = X.to(device), y.to(device)
torch.cuda.synchronize()
t = time.time()
for _ in range(20):
    opt.zero_grad()
    loss_fn(m(X), y).backward()
    opt.step()
torch.cuda.synchronize()
print("GPU only, 20 batches:", time.time() - t)

data only, 50 batches: 0.42010021209716797
GPU only, 20 batches: 4.529326915740967


In [15]:
import sys
sys.path.insert(0, "/kaggle/working/repo/src")
SPLITS_DIR = Path("/kaggle/working/repo/splits")

In [3]:
!find / -name "best_seed*.pt" 2>/dev/null
!ls /kaggle/working